# 2.2 Individual reaction examples

Select illustrative reactions from the ensemble predictions exported by notebook 2.1. Each row represents one reaction. The selection is post hoc; overall performance is reported for the full external set in notebook 2.1.

The current thresholds are an aligned-transfer absolute error ≤ 0.30 kcal/mol and both control errors ≥ 0.40 kcal/mol. Up to 10 candidates are selected, with optional manual row indices. The selected reactions are illustrative examples. No models are fitted here.


In [1]:
from pathlib import Path
from collections import OrderedDict
import hashlib
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT_OVERRIDE = None  # e.g. Path('/path/to/ru-mechanism-transfer')
ALIGNED_MAX_ABS_ERROR = 0.30
TARGET_MIN_ABS_ERROR = 0.40
MISMATCHED_MIN_ABS_ERROR = 0.40
N_EXAMPLES = 10
# Optional zero-based external_row_index values from the candidate table.
MANUAL_ROW_INDICES = []
EXPORT_STRUCTURE_SVGS = True
SHOW_TEXT = False  # Hide structure titles; retain atom labels.

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'results/2_1_recent_aho_external_validation/tables/external_predictions.csv').is_file()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Outputs from notebook 2.1 were not found. Run 2.1 or set PROJECT_ROOT_OVERRIDE.')
INPUT_DIR = PROJECT_ROOT / 'results/2_1_recent_aho_external_validation/tables'
PREDICTION_PATH = INPUT_DIR / 'external_predictions.csv'
METRICS_PATH = INPUT_DIR / 'external_metrics_all_selected_models.csv'
OUT_DIR = PROJECT_ROOT / 'results/2_2_individual_reaction_examples'
TABLE_DIR = OUT_DIR / 'tables'
STRUCTURE_DIR = OUT_DIR / 'structures'
for p in [TABLE_DIR, STRUCTURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
SCENARIOS = OrderedDict([
    ('target_only', 'Target-only'),
    ('mechanism_mismatched', 'Mechanism-mismatched transfer'),
    ('mechanism_aligned', 'Mechanism-aligned transfer'),
])
for value in [ALIGNED_MAX_ABS_ERROR, TARGET_MIN_ABS_ERROR, MISMATCHED_MIN_ABS_ERROR]:
    if not np.isfinite(value) or value < 0:
        raise ValueError('Error thresholds must be finite and non-negative.')
if not isinstance(N_EXAMPLES, int) or N_EXAMPLES < 1:
    raise ValueError('N_EXAMPLES must be a positive integer.')
print('Read:', INPUT_DIR)
print('Write:', OUT_DIR)


Read: /home/syz/ru-mechanism-transfer/results/2_1_recent_aho_external_validation/tables
Write: /home/syz/ru-mechanism-transfer/results/2_2_individual_reaction_examples


## 1. Load ensemble predictions

Match ensemble prediction columns to the metric table and verify the recomputed metrics.


In [2]:
def safe_name(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')

for p in [PREDICTION_PATH, METRICS_PATH]:
    if not p.is_file():
        raise FileNotFoundError(f'Run notebook 2.1 first; missing file: {p}')
raw = pd.read_csv(PREDICTION_PATH).reset_index(drop=True)
metric_table = pd.read_csv(METRICS_PATH)
for table, cols in [(raw, ['y_true','Reactant SMILES','Product SMILES','Ligand SMILES']),
                    (metric_table, ['result_type','scenario','result_name','MAE','RMSE','R2','PearsonR'])]:
    missing = set(cols)-set(table.columns)
    if missing:
        raise ValueError(f'Missing input columns: {sorted(missing)}')
if len(raw) < 2:
    raise ValueError('Insufficient external prediction records.')
work = raw.copy()
# This positional identity is tied to the exact CSV hash saved in the manifest.
if 'external_row_index' in work.columns:
    work = work.rename(columns={'external_row_index':'upstream_external_row_index'})
work.insert(0, 'external_row_index', np.arange(len(work)))
work['y_true'] = pd.to_numeric(work.y_true, errors='raise')
if not np.isfinite(work.y_true).all():
    raise ValueError('Observed values contain non-finite entries.')
AUDIT_ROWS = []
for key, scenario in SCENARIOS.items():
    matched = metric_table.loc[metric_table.result_type.eq('ensemble') & metric_table.scenario.eq(scenario)]
    if len(matched) != 1:
        raise ValueError(f'{scenario}: Expected one ensemble metric row; found {len(matched)}')
    row = matched.iloc[0]
    column = 'pred_' + safe_name(row.result_name)
    if column not in raw:
        raise KeyError(column)
    pred = pd.to_numeric(raw[column], errors='raise').to_numpy()
    y = work.y_true.to_numpy()
    if not np.isfinite(pred).all():
        raise ValueError(f'{scenario}: Invalid predictions')
    err = pred-y
    sst = np.sum((y-y.mean())**2)
    calculated = {'MAE':np.mean(abs(err)), 'RMSE':np.sqrt(np.mean(err**2)),
                  'R2':1-np.sum(err**2)/sst if sst>0 else (1. if np.all(err==0) else 0.),
                  'PearsonR':np.corrcoef(y,pred)[0,1] if np.std(y)>1e-12 and np.std(pred)>1e-12 else np.nan}
    for name,value in calculated.items():
        if not np.isclose(value,float(row[name]),rtol=1e-6,atol=1e-6,equal_nan=True):
            raise ValueError(f'{scenario}/{name}: Predictions do not match the metric table. Rerun notebook 2.1.')
    work['prediction_'+key] = pred
    work['abs_error_'+key] = abs(err)
    AUDIT_ROWS.append({'scenario_key':key,'result_name':row.result_name,'prediction_column':column,**calculated})
PREDICTION_AUDIT = pd.DataFrame(AUDIT_ROWS)
PREDICTION_AUDIT.to_csv(TABLE_DIR / 'prediction_column_and_metric_audit.csv',index=False)
display(PREDICTION_AUDIT)


,scenario_key,result_name,prediction_column,MAE,RMSE,R2,PearsonR
0,target_only,Target-only | 3 MACCS + 3 SPOC ensemble mean,pred_target_only_3_maccs_3_spoc_ensemble_mean,0.747288,0.857183,0.187802,0.532516
1,mechanism_mismatched,Mechanism-mismatched transfer | 3 MACCS + 3 SP...,pred_mechanism_mismatched_transfer_3_maccs_3_s...,0.770646,0.890914,0.122623,0.382323
2,mechanism_aligned,Mechanism-aligned transfer | 3 MACCS + 3 SPOC ...,pred_mechanism_aligned_transfer_3_maccs_3_spoc...,0.650289,0.745544,0.385587,0.704195


## 2. Rank eligible reactions

Rank candidates by the improvement over the better of the two controls, followed by the aligned-transfer absolute error.


In [3]:
work['aligned_pass'] = work.abs_error_mechanism_aligned <= ALIGNED_MAX_ABS_ERROR
work['target_pass'] = work.abs_error_target_only >= TARGET_MIN_ABS_ERROR
work['mismatched_pass'] = work.abs_error_mechanism_mismatched >= MISMATCHED_MIN_ABS_ERROR
work['eligible'] = work[['aligned_pass','target_pass','mismatched_pass']].all(axis=1)
work['improvement_vs_target'] = work.abs_error_target_only-work.abs_error_mechanism_aligned
work['improvement_vs_mismatched'] = work.abs_error_mechanism_mismatched-work.abs_error_mechanism_aligned
work['min_improvement'] = work[['improvement_vs_target','improvement_vs_mismatched']].min(axis=1)
CANDIDATES = work.loc[work.eligible].sort_values(
    ['min_improvement','abs_error_mechanism_aligned','external_row_index'],ascending=[False,True,True]).copy()
CANDIDATES.insert(0,'candidate_rank',np.arange(1,len(CANDIDATES)+1))
work.to_csv(TABLE_DIR / 'all_external_reactions_scored.csv',index=False)
CANDIDATES.to_csv(TABLE_DIR / 'all_eligible_candidates.csv',index=False)
VIEW_COLS = ['external_row_index','y_true','prediction_target_only','prediction_mechanism_mismatched',
             'prediction_mechanism_aligned','abs_error_target_only','abs_error_mechanism_mismatched',
             'abs_error_mechanism_aligned','min_improvement']
CONTEXT_COLS = [c for c in ['Reaction Year','DOI','doi','Reference','Reactant SMILES','Product SMILES',
    'Ligand SMILES','Metal','Solvent SMILES','Additive SMILES','Pressure/atm','Temperature/C','S/C'] if c in work]
print(f'Eligible reactions: {len(CANDIDATES)} / {len(work)}; thresholds are fixed.')
display(CANDIDATES[['candidate_rank',*VIEW_COLS,*CONTEXT_COLS]])


Eligible reactions: 4 / 103; thresholds are fixed.


,candidate_rank,external_row_index,y_true,prediction_target_only,prediction_mechanism_mismatched,prediction_mechanism_aligned,abs_error_target_only,abs_error_mechanism_mismatched,abs_error_mechanism_aligned,min_improvement,Reaction Year,Reactant SMILES,Product SMILES,Ligand SMILES,Metal,Solvent SMILES,Additive SMILES,Pressure/atm,Temperature/C,S/C
87,1,87,1.075530,2.418386,2.729990,1.373919,1.342856,1.654461,0.298389,1.044467,2022,C=C(OCC)c1c(C(=O)OCC)cnn2ccnc12,CCOC(=O)c1cnn2ccnc2c1[C@@H](C)OCC,COc1ccc(C(N)(c2ccc(OC)cc2)[C@H](N)C(C)C)cc1.Cc...,Ru,CCO,NaN,50.0,25.0,100.0
81,2,81,1.180482,1.620977,1.788873,1.275345,0.440495,0.608390,0.094863,0.345632,2022,C=C(OCC)c1c(C(=O)OCC)cnn2ccnc12,CCOC(=O)c1cnn2ccnc2c1[C@@H](C)OCC,Cc1ccc(C(C)C)cc1.c1ccc(P(c2ccccc2)c2ccc3ccccc3...,Ru,CCO,NaN,50.0,25.0,100.0
71,3,71,1.744532,2.163790,2.248894,1.849890,0.419258,0.504362,0.105358,0.313900,2024,C=C(CC(=O)c1ccccc1)C(=O)O,C[C@@H](CC(=O)c1ccccc1)C(=O)O,CC(C)(C)[C@H]1COC([C]23[CH]4[CH]5[CH]6[C]2(P(c...,Ru,CO,[K+].[OH-],50.0,25.0,100.0
85,4,85,1.100512,1.601361,1.964225,1.334105,0.500849,0.863713,0.233593,0.267256,2022,C=C(OCC)c1c(C(=O)OCC)cnn2ccnc12,CCOC(=O)c1cnn2ccnc2c1[C@@H](C)OCC,Cc1cc(C)cc(P(c2cc(C)cc(C)c2)c2ccc3c(c2-c2c(P(c...,Ru,CCO,NaN,50.0,25.0,100.0


## 3. Select examples

Prefer distinct substrates, then fill remaining positions by rank. Manually specified rows must satisfy the same thresholds. Retain row identifiers and reaction conditions.


In [4]:
# RDKit is already a dependency of 2_1. No averaging or structural deduplication is performed.
from rdkit import Chem

def substrate_key(smiles):
    if pd.isna(smiles):
        return ''
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol, isomericSmiles=True) if mol is not None else str(smiles).strip()

if MANUAL_ROW_INDICES:
    ids = list(MANUAL_ROW_INDICES)
    if len(ids)!=len(set(ids)) or not all(isinstance(i,(int,np.integer)) for i in ids):
        raise ValueError('Manual row indices must be unique integers.')
    allowed = set(CANDIDATES.external_row_index)
    if not set(ids).issubset(allowed):
        raise ValueError(f'Row indices absent or outside the selection criteria: {set(ids)-allowed}')
else:
    ids, seen = [], set()
    for row in CANDIDATES.itertuples(index=False):
        key = substrate_key(CANDIDATES.loc[CANDIDATES.external_row_index.eq(row.external_row_index),'Reactant SMILES'].iloc[0])
        if key not in seen:
            ids.append(row.external_row_index); seen.add(key)
        if len(ids)>=N_EXAMPLES:
            break
    for idx in CANDIDATES.external_row_index:
        if len(ids)>=N_EXAMPLES:
            break
        if idx not in ids:
            ids.append(idx)
SELECTED = CANDIDATES.set_index('external_row_index').reindex(ids).reset_index()
SELECTED.insert(0,'example_id',[f'Example_{i:02d}' for i in range(1,len(SELECTED)+1)])
SELECTED.to_csv(TABLE_DIR / 'selected_examples_full_records.csv',index=False)
SELECTED[['example_id',*VIEW_COLS,*CONTEXT_COLS]].to_csv(TABLE_DIR / 'selected_examples_for_layout.csv',index=False)
comparison=[]
for _, row in SELECTED.iterrows():
    for key,label in SCENARIOS.items():
        comparison.append({'example_id':row.example_id,'external_row_index':row.external_row_index,
            'scenario':label,'observed_ddG':row.y_true,'predicted_ddG':row['prediction_'+key],
            'absolute_error':row['abs_error_'+key]})
COMPARISON = pd.DataFrame(comparison,columns=['example_id','external_row_index','scenario','observed_ddG','predicted_ddG','absolute_error'])
COMPARISON.to_csv(TABLE_DIR / 'selected_examples_prediction_comparison.csv',index=False)
display(SELECTED[['example_id',*VIEW_COLS,*CONTEXT_COLS]])
print('Number of selected examples: ',len(SELECTED))
if len(SELECTED)==0:
    print('No eligible examples. The full scoring table has been saved.')


,example_id,external_row_index,y_true,prediction_target_only,prediction_mechanism_mismatched,prediction_mechanism_aligned,abs_error_target_only,abs_error_mechanism_mismatched,abs_error_mechanism_aligned,min_improvement,Reaction Year,Reactant SMILES,Product SMILES,Ligand SMILES,Metal,Solvent SMILES,Additive SMILES,Pressure/atm,Temperature/C,S/C
0,Example_01,87,1.075530,2.418386,2.729990,1.373919,1.342856,1.654461,0.298389,1.044467,2022,C=C(OCC)c1c(C(=O)OCC)cnn2ccnc12,CCOC(=O)c1cnn2ccnc2c1[C@@H](C)OCC,COc1ccc(C(N)(c2ccc(OC)cc2)[C@H](N)C(C)C)cc1.Cc...,Ru,CCO,NaN,50.0,25.0,100.0
1,Example_02,71,1.744532,2.163790,2.248894,1.849890,0.419258,0.504362,0.105358,0.313900,2024,C=C(CC(=O)c1ccccc1)C(=O)O,C[C@@H](CC(=O)c1ccccc1)C(=O)O,CC(C)(C)[C@H]1COC([C]23[CH]4[CH]5[CH]6[C]2(P(c...,Ru,CO,[K+].[OH-],50.0,25.0,100.0
2,Example_03,81,1.180482,1.620977,1.788873,1.275345,0.440495,0.608390,0.094863,0.345632,2022,C=C(OCC)c1c(C(=O)OCC)cnn2ccnc12,CCOC(=O)c1cnn2ccnc2c1[C@@H](C)OCC,Cc1ccc(C(C)C)cc1.c1ccc(P(c2ccccc2)c2ccc3ccccc3...,Ru,CCO,NaN,50.0,25.0,100.0
3,Example_04,85,1.100512,1.601361,1.964225,1.334105,0.500849,0.863713,0.233593,0.267256,2022,C=C(OCC)c1c(C(=O)OCC)cnn2ccnc12,CCOC(=O)c1cnn2ccnc2c1[C@@H](C)OCC,Cc1cc(C)cc(P(c2cc(C)cc(C)c2)c2ccc3c(c2-c2c(P(c...,Ru,CCO,NaN,50.0,25.0,100.0


Number of selected examples:  4


## 4. Export structures and records

Export substrate, product, and ligand-representation SVGs for figure assembly. Ligand SMILES may omit metal coordination. Failed structure parsing is recorded in the export table. `structure_export_manifest.csv` identifies the files from the current run.


In [5]:
from rdkit.Chem.Draw import rdMolDraw2D
structure_log=[]
if EXPORT_STRUCTURE_SVGS:
    for _,row in SELECTED.iterrows():
        for column,tag in [('Reactant SMILES','substrate'),('Product SMILES','product'),('Ligand SMILES','ligand_representation')]:
            smi=row[column]
            mol=Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
            record={'example_id':row.example_id,'external_row_index':row.external_row_index,'component':tag,'smiles':smi}
            if mol is None:
                structure_log.append({**record,'status':'invalid_smiles','file':''});continue
            path=STRUCTURE_DIR / f'{row.example_id}_row{int(row.external_row_index)}_{tag}.svg'
            drawer=rdMolDraw2D.MolDraw2DSVG(600,400)
            drawer.drawOptions().useBWAtomPalette()
            drawer.DrawMolecule(mol,legend=f'{row.example_id}: {tag}' if SHOW_TEXT else '')
            drawer.FinishDrawing()
            path.write_text(drawer.GetDrawingText(),encoding='utf-8')
            structure_log.append({**record,'status':'saved','file':str(path)})
pd.DataFrame(structure_log,columns=['example_id','external_row_index','component','smiles','status','file']).to_csv(
    TABLE_DIR / 'structure_export_manifest.csv',index=False)
manifest={'analysis':'post hoc illustrative individual reaction selection; no model fitting',
    'thresholds':{'aligned_max':ALIGNED_MAX_ABS_ERROR,'target_min':TARGET_MIN_ABS_ERROR,'mismatched_min':MISMATCHED_MIN_ABS_ERROR},
    'n_external':len(work),'n_eligible':len(CANDIDATES),'selected_row_indices':[int(i) for i in ids],
    'inputs':{str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in [PREDICTION_PATH,METRICS_PATH]}}
(OUT_DIR / 'selection_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
assert SELECTED.external_row_index.is_unique
assert set(SELECTED.external_row_index).issubset(set(work.loc[work.eligible,'external_row_index']))
print('Output directory: ',OUT_DIR)


Output directory:  /home/syz/ru-mechanism-transfer/results/2_2_individual_reaction_examples
